In [2]:
!pip install -q transformers==4.46 torch==2.5.0 accelerate== 1.0.1


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: 'accelerate==': Expected semicolon (after name with no version specifier) or end
    accelerate==
              ^


In [3]:
import os, random, textwrap
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


In [4]:
BASE_MODEL_ID = "distilgpt2"
base_generator = pipeline(task="text-generation", model= BASE_MODEL_ID, device=0 if DEVICE=="cuda" else -1)
print("Loaded:", BASE_MODEL_ID)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

c:\Users\908136\Workspace\Lab0\myvenv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\908136\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\908136\Workspace\Lab0\myvenv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded: distilgpt2


In [9]:
def show(title,text):
    print("\n" + "="*90)
    print(title)
    print("-"*90)
    print(textwrap.fill(text, width=100))
    print("-"*90)

def safe_generate(gen_pipeline,prompt, max_new_tokens=80, temperature=0.7, top_p=0.95, do_sample=True):

    try:
        tokenizer = gen_pipeline.tokenizer
        pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    except Exception:
        pad_token_id= None

    kwargs= dict(max_new_tokens=max_new_tokens, temperature= temperature, top_p=top_p, do_sample=do_sample, return_full_text=True)

    if pad_token_id is not None:
        kwargs["pad_token_id"] = pad_token_id

    output= gen_pipeline(prompt, **kwargs)[0]["generated_text"]
    return output

def try_prompt(label,gen_pipeline,prompt, **gen_kwargs):
    out= safe_generate(gen_pipeline, prompt, **gen_kwargs)
    show(f"{label}  PROMPT", prompt)
    show(f"{label}  OUTPUT", out)


In [10]:
GEN= base_generator
prompt= "Summarize in one sentence: Prompt engineering helps models follow human intent clearly."
try_prompt("SMOKE TEST", GEN, prompt, max_new_tokens=40, temperature=0.7)

[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



SMOKE TEST  PROMPT
------------------------------------------------------------------------------------------
Summarize in one sentence: Prompt engineering helps models follow human intent clearly.
------------------------------------------------------------------------------------------

SMOKE TEST  OUTPUT
------------------------------------------------------------------------------------------
Summarize in one sentence: Prompt engineering helps models follow human intent clearly.     This is
my first post on the web, and the first one of the two parts on the web. I want to get people to
understand the problem of this problem.
------------------------------------------------------------------------------------------
